# Assignment 1 - LLM Evaluation: Product Description Generation

**Course:** AI Performance Engineering  
**Due Date:** April 5, 2026

This notebook contains the complete solution for Assignment 1, covering:
1. Rubric definition
2. Description generation
3. Manual evaluation
4. Improvement cycle
5. Judge model creation
6. Judge analysis and comparison

---
## Task 1: Define Your Rubric (15 points)

Before generating or evaluating anything, we need a clear, repeatable scoring framework.

### 1.1 Criterion Definitions

For each criterion, we define explicit standards for **good**, **ok**, and **bad** ratings. These are stored in a structured dictionary for programmatic evaluation.

### 1.2 Criterion Thresholds Dictionary

In [4]:
# Criterion thresholds for automated evaluation
CRITERION_THRESHOLDS = {
    "fluency": {
        "criteria_type": "quality",
        "check": "manual",
        "good": {
            "description": "Natural, smooth sentences with varied structure. Easy to read aloud. No awkward phrasing or repetition.",
        },
        "ok": {
            "description": "Mostly natural but with minor awkwardness (e.g., one slightly repetitive phrase or choppy transition).",
        },
        "bad": {
            "description": "Multiple awkward phrases, unnatural word order, or repetitive structure that disrupts readability.",
        },
    },
    "grammar": {
        "criteria_type": "quality",
        "check": "manual",
        "good": {
            "description": "Zero spelling or punctuation errors. Proper sentence structure throughout.",
        },
        "ok": {
            "description": "One minor error (e.g., missing comma, minor typo) that doesn't affect comprehension.",
        },
        "bad": {
            "description": "Multiple errors or one major error (e.g., subject-verb disagreement, misspelled product name).",
        },
    },
    "tone": {
        "criteria_type": "quality",
        "check": "manual",
        "good": {
            "description": "Consistently friendly, credible sales voice. Enthusiastic without being pushy. Professional language appropriate for e-commerce.",
        },
        "ok": {
            "description": "Generally appropriate tone but with one instance of overly casual language, excessive hype, or slightly flat delivery.",
        },
        "bad": {
            "description": "Inappropriate tone (too formal/technical, too casual, or overly aggressive sales language). Multiple tone inconsistencies.",
        },
    },
    "length": {
        "criteria_type": "quality",
        "check": "automatic",
        "good": {
            "description": "50-90 words (inclusive)",
            "ranges": [(50, 90)],
        },
        "ok": {
            "description": "40-49 words OR 91-110 words",
            "ranges": [(40, 49), (91, 110)],
        },
        "bad": {
            "description": "Fewer than 40 words OR more than 110 words",
            "ranges": [(0, 39), (111, 999999)],
        },
    },
    "grounding": {
        "criteria_type": "quality",
        "check": "manual",
        "good": {
            "description": "All information comes directly from provided data (name, attributes, material, warranty). No fabricated features or specifications.",
        },
        "ok": {
            "description": "Minor embellishment that's reasonable inference (e.g., 'sleek design' when material is 'aluminum') but no false claims.",
        },
        "bad": {
            "description": "Contains fabricated information, incorrect specifications, or claims not supported by the provided data.",
        },
    },
    "latency": {
        "criteria_type": "objective",
        "check": "automatic",
        "good": {
            "description": "≤ 5000ms (5 seconds)",
            "ranges": [(0, 5000)],
        },
        "ok": {
            "description": "5001-10000ms (5-10 seconds)",
            "ranges": [(5001, 10000)],
        },
        "bad": {
            "description": "> 10000ms (10+ seconds)",
            "ranges": [(10001, 999999)],
        },
    },
    "cost": {
        "criteria_type": "objective",
        "check": "automatic",
        "good": {
            "description": "≤ $0.01 per description",
            "ranges": [(0, 0.01)],
        },
        "ok": {
            "description": "$0.011-$0.05 per description",
            "ranges": [(0.011, 0.05)],
        },
        "bad": {
            "description": "> $0.05 per description",
            "ranges": [(0.051, 999999)],
        },
    },
}


def evaluate_criterion(criterion_name: str, value: float) -> str:
    """
    Generic evaluation function for any criterion with numeric ranges.

    Args:
        criterion_name: Name of the criterion (e.g., 'length', 'latency', 'cost')
        value: Numeric value to evaluate

    Returns:
        'good', 'ok', or 'bad'
    """
    thresholds = CRITERION_THRESHOLDS[criterion_name]

    for rating in ["good", "ok", "bad"]:
        for min_val, max_val in thresholds[rating]["ranges"]:
            if min_val <= value <= max_val:
                return rating

    return "bad"


def evaluate_length(word_count: int) -> str:
    """Evaluate length criterion based on word count using CRITERION_THRESHOLDS."""
    return evaluate_criterion("length", word_count)


def evaluate_latency(latency_ms: float) -> str:
    """Evaluate latency criterion based on milliseconds using CRITERION_THRESHOLDS."""
    return evaluate_criterion("latency", latency_ms)


def evaluate_cost(cost_usd: float) -> str:
    """Evaluate cost criterion based on USD amount using CRITERION_THRESHOLDS."""
    return evaluate_criterion("cost", cost_usd)


def calculate_pass_fail(ratings: dict) -> str:
    """
    Calculate pass/fail based on ratings.

    Args:
        ratings: dict with keys ['fluency', 'grammar', 'tone', 'length', 'grounding', 'latency', 'cost']
                 values: 'good', 'ok', or 'bad'

    Returns:
        'pass' or 'fail'

    Rules:
        - Automatic failure if grounding, grammar, or length is "bad"
        - Pass requires: ≥4 good, ≤1 bad, and ≥5 acceptable (good or ok)
    """
    # Go/no-go rules - automatic failure conditions
    if ratings["grounding"] == "bad":
        return "fail"
    if ratings["grammar"] == "bad":
        return "fail"
    if ratings["length"] == "bad":
        return "fail"

    # Cumulative pass bar - count ratings by type
    good_count = sum(1 for v in ratings.values() if v == "good")
    ok_count = sum(1 for v in ratings.values() if v == "ok")
    bad_count = sum(1 for v in ratings.values() if v == "bad")

    # Calculate combined acceptable ratings
    good_or_ok_count = good_count + ok_count

    # Must have: ≥4 good, ≤1 bad, and ≥5 acceptable (good or ok)
    if (good_count >= 4) and (bad_count <= 1) and (good_or_ok_count >= 5):
        return "pass"
    else:
        return "fail"

---
## Task 2: Generate Descriptions for Every Product (20 points)

Generate product descriptions using a language model from Nebius Token Factory.

In [5]:
# Import required libraries
import os
import time

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables
load_dotenv()

# Constants
NEBIUS_API_BASE_URL = "https://api.tokenfactory.nebius.com/v1/"
# split criteria into quality and objective
QUALITY_CRITERIA = [
    k for k, v in CRITERION_THRESHOLDS.items() if v.get("criteria_type") == "quality"
]
OBJECTIVE_CRITERIA = [
    k for k, v in CRITERION_THRESHOLDS.items() if v.get("criteria_type") == "objective"
]
EVALUATION_CRITERIA = list[str](CRITERION_THRESHOLDS.keys())

OUTPUT_EXCEL_PATH = "assignment_01.xlsx"

# Initialize OpenAI client
client = OpenAI(base_url=NEBIUS_API_BASE_URL, api_key=os.environ.get("NEBIUS_API_KEY"))

print(EVALUATION_CRITERIA)

['fluency', 'grammar', 'tone', 'length', 'grounding', 'latency', 'cost']


In [6]:
# Load the product dataset
df = pd.read_csv("Assignment_01_product_dataset.csv")
print(f"Loaded {len(df)} products")
df.head(1)

Loaded 50 products


,product_name,Product_attribute_list,material,warranty
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty


### 2.1 System Prompt

Design a prompt that instructs the model to generate persuasive 50-90 word product descriptions.

In [7]:
SYSTEM_PROMPT = """
You are an expert e-commerce copywriter. Your task is to write persuasive product descriptions for online shoppers.

Requirements:
- Length: Exactly 50-90 words
- Tone: Friendly, credible, and enthusiastic (but not pushy)
- Content: Use ONLY the provided product information - do not fabricate features
- Style: Natural, easy-to-read sentences with varied structure
- Grammar: Perfect spelling and punctuation

Focus on benefits and appeal to the target customer. Make them want to buy!

OUTPUT: Provide only the product description text. Do not include any preamble, explanation, or additional commentary.
""".strip()


def create_user_prompt(
    product_name: str, attributes: str, material: str, warranty: str
) -> str:
    return f"""Product Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

Write a persuasive product description (50-90 words)."""

### 2.2 Model Selection

Choose one model from Nebius Token Factory:
- Gemma-2-9b-it
- Meta-Llama-3.1-8B-Instruct

In [8]:
# Choosing the model
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # or "google/gemma-2-9b-it"


def generate_description(
    product_name: str,
    attributes: str,
    material: str,
    warranty: str,
    *,
    model_name: str = None,
    system_prompt: str = None,
    temperature: float = None,
    max_completion_tokens: int = None,
) -> dict:
    """
    Generate a product description and collect metrics.

    Args:
        product_name: Product name
        attributes: Product attributes
        material: Product material
        warranty: Product warranty
        model_name: Model to use (defaults to MODEL_NAME)
        system_prompt: System prompt (defaults to SYSTEM_PROMPT) (optional)
        temperature: Sampling temperature (optional)
        max_completion_tokens: Max tokens to generate

    Returns:
        dict with keys: generated_description, latency_ms, input_tokens, output_tokens
    """
    user_prompt = create_user_prompt(product_name, attributes, material, warranty)

    start_time = time.time()

    # Build request with optional parameters
    request_kwargs = {
        "model": model_name if model_name is not None else MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    }

    if max_completion_tokens is not None:
        request_kwargs["max_completion_tokens"] = max_completion_tokens

    if temperature is not None:
        request_kwargs["temperature"] = temperature

    response = client.chat.completions.create(**request_kwargs)

    end_time = time.time()
    latency_ms = int((end_time - start_time) * 1000)

    return {
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms": latency_ms,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    }

### 2.3 Generate Descriptions for All Products

In [9]:
def generate_descriptions_for_df(
    source_df: pd.DataFrame,
    *,
    n_rows: int = None,
    model_name: str = None,
    system_prompt: str = None,
    temperature: float = None,
    max_completion_tokens: int = None,
    sleep_s: float = 0.5,
) -> pd.DataFrame:
    """
    Generate descriptions for multiple products in a DataFrame.

    Args:
        source_df: DataFrame with product data (must have columns:
                   product_name, Product_attribute_list, material, warranty)
        n_rows: Number of rows to process (None = all rows)
        model_name: Model to use (defaults to MODEL_NAME)
        system_prompt: System prompt (defaults to SYSTEM_PROMPT)
        temperature: Sampling temperature (optional)
        max_completion_tokens: Max tokens to generate
        sleep_s: Sleep duration between requests to avoid rate limiting

    Returns:
        DataFrame with original columns plus generated_description, latency_ms,
        input_tokens, output_tokens
    """
    subset = source_df.head(n_rows).copy() if n_rows else source_df.copy()
    results = []

    for idx, row in subset.iterrows():
        print(f"Processing {idx + 1}/{len(subset)}: {row['product_name']}")

        out = generate_description(
            product_name=row["product_name"],
            attributes=row["Product_attribute_list"],
            material=row["material"],
            warranty=row["warranty"],
            model_name=model_name,
            system_prompt=system_prompt,
            temperature=temperature,
            max_completion_tokens=max_completion_tokens,
        )

        results.append({**row.to_dict(), **out})
        time.sleep(sleep_s)

    print("\nGeneration complete!")
    return pd.DataFrame(results)

In [13]:
if not os.path.exists(OUTPUT_EXCEL_PATH):
    print("File does not exist, running generation")
    # Run generation for all products using the reusable helper
    results_df = generate_descriptions_for_df(
        df, n_rows=None, system_prompt=SYSTEM_PROMPT
    )

    # Add blank columns for evaluation criteria
    for criterion in EVALUATION_CRITERIA:
        results_df[criterion] = ""

    results_df["final_score"] = ""

    # Save to Excel
    results_df.to_excel(OUTPUT_EXCEL_PATH, index=False)
    print(f"Saved results to {OUTPUT_EXCEL_PATH}")

    # Display summary
    print(f"\nGenerated {len(results_df)} descriptions")
    print(f"Average latency: {results_df['latency_ms'].mean():.0f}ms")
    print(f"Average input tokens: {results_df['input_tokens'].mean():.0f}")
    print(f"Average output tokens: {results_df['output_tokens'].mean():.0f}")
else:
    print("File exists, loading results")
    # Same shape as the generation branch (list of row dicts)
    results_df = pd.read_excel(OUTPUT_EXCEL_PATH)

File exists, loading results


In [14]:
results_df.head(1)

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,word_counts,cost_value,fluency,grammar,tone,length,grounding,latency,cost,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,Take your mobile experience to the next level ...,4720,200,110,87.0,0.000011,ok,good,ok,good,ok,good,good,pass


---
## Task 3: Manual (Human) Evaluation (10 points)

Manually evaluate 10-15 products using the rubric defined in Task 1.

### 3.1 Add Cost Column

Calculate the cost per description based on token usage and model pricing.

In [15]:
# Fetch pricing dynamically from Nebius API
def get_model_pricing(model_name: str) -> tuple[float, float]:
    """
    Fetch pricing for a specific model from Nebius Token Factory API.

    Args:
        model_name: The model identifier (e.g., "meta-llama/Meta-Llama-3.1-8B-Instruct")

    Returns:
        Tuple of (input_price_per_1k_tokens, output_price_per_1k_tokens) in USD
    """
    try:
        # List all models with verbose=true to get pricing information
        # Note: The OpenAI client doesn't support query params directly,
        # so we need to make a raw HTTP request
        import requests

        response = requests.get(
            f"{NEBIUS_API_BASE_URL}models?verbose=true",
            headers={"Authorization": f"Bearer {os.environ.get('NEBIUS_API_KEY')}"},
        )
        response.raise_for_status()
        models_data = response.json()

        # Find the specific model
        for model in models_data.get("data", []):
            if model.get("id") == model_name:
                pricing = model.get("pricing", {})

                # Pricing values are strings representing price per token
                # Convert to float and then to per 1K tokens
                input_price_per_token = float(pricing.get("prompt", "0"))
                output_price_per_token = float(pricing.get("completion", "0"))

                input_price_per_1k = input_price_per_token * 1000
                output_price_per_1k = output_price_per_token * 1000

                print(f"✓ Fetched pricing for {model_name}:")
                print(f"  Input:  ${input_price_per_1k:.6f} per 1K tokens")
                print(f"  Output: ${output_price_per_1k:.6f} per 1K tokens")
                return input_price_per_1k, output_price_per_1k

        # Model not found
        raise ValueError(f"Model not found: {model_name}")

    except Exception as e:
        print(f"⚠ Warning: Could not fetch pricing from API: {e}")
        print("Using fallback pricing values for meta-llama/Meta-Llama-3.1-8B-Instruct")
        # Fallback: $0.02/1M input, $0.06/1M output = $0.00002/1K, $0.00006/1K
        # NOTE: This is the pricing for the model used in the assignment ("meta-llama/Meta-Llama-3.1-8B-Instruct")
        return (0.00002, 0.00006)


# Get pricing for the chosen model
PRICE_PER_1K_INPUT_TOKENS, PRICE_PER_1K_OUTPUT_TOKENS = get_model_pricing(MODEL_NAME)

# Calculate cost (numeric; overwrites any placeholder in the cost column)
results_df["cost_value"] = (
    results_df["input_tokens"] / 1000 * PRICE_PER_1K_INPUT_TOKENS
) + (results_df["output_tokens"] / 1000 * PRICE_PER_1K_OUTPUT_TOKENS)

print(f"\nAverage cost per description: ${results_df['cost_value'].mean():.6f}")
print(
    f"Total cost for {len(results_df)} descriptions: ${results_df['cost_value'].sum():.6f}"
)

results_df = results_df.fillna("")
# Save the results (na_rep="" keeps future reads aligned with empty-string placeholders)
results_df.to_excel(OUTPUT_EXCEL_PATH, index=False, na_rep="")
results_df.to_csv("dataset_after_generating_descriptions.csv", index=False)

✓ Fetched pricing for meta-llama/Meta-Llama-3.1-8B-Instruct:
  Input:  $0.000020 per 1K tokens
  Output: $0.000060 per 1K tokens

Average cost per description: $0.000010
Total cost for 50 descriptions: $0.000515


### 3.2 Manual Evaluation Instructions

**Manually evaluate 10-15 products**

1. Open OUTPUT_EXCEL_PATH (`assignment_01.xlsx`) in Excel
2. Select 10-15 diverse products
3. For each selected product, rate each criterion (fluency, grammar, tone, length, grounding, latency, cost) as:
   - `good`
   - `ok`
   - `bad`
4. Use the rubric definitions from Task 1

After completing manual evaluation, run the cell below to load and analyze your scores.

In [69]:
def evaluate_automatic_criteria(df: pd.DataFrame, n_rows: int = None) -> pd.DataFrame:
    """
    Evaluate automatic criteria (length, latency, cost).

    Args:
        excel_path: Path to the Excel file
        n_rows: Number of rows to evaluate (None = all rows)

    Returns:
        DataFrame with automatic evaluations
    """
    # Read Excel file
    # Empty Excel cells are often inferred as float64; rubric values are strings.
    for col in ("length", "latency"):
        if col in df.columns:
            df[col] = df[col].astype(object)

    # Limit to first n rows if specified
    rows_to_eval = df.head(n_rows) if n_rows else df

    # Evaluate length (count words in generated_description)
    word_counts = rows_to_eval["generated_description"].apply(
        lambda x: len(str(x).split())
    )
    # evaluate_length / evaluate_latency / evaluate_cost each take one numeric arg
    # and call evaluate_criterion internally (same rubric as evaluate_criterion("…", x))
    df.loc[rows_to_eval.index, "word_counts"] = word_counts
    df.loc[rows_to_eval.index, "length"] = word_counts.map(evaluate_length)

    df.loc[rows_to_eval.index, "latency"] = rows_to_eval["latency_ms"].map(
        evaluate_latency
    )

    df.loc[rows_to_eval.index, "cost"] = df.loc[rows_to_eval.index, "cost_value"].apply(
        lambda x: evaluate_cost(x) if isinstance(x, (int, float)) else x
    )

    return df


# Evaluate first 15 rows and save
automatic_eval_df = evaluate_automatic_criteria(results_df, n_rows=10).fillna("")[
    [
        "product_name",
        "Product_attribute_list",
        "material",
        "warranty",
        "generated_description",
        "latency_ms",
        "input_tokens",
        "output_tokens",
        "word_counts",
        "cost_value",
        *EVALUATION_CRITERIA,
        "final_score",
    ]
]

print("\nRows after automatic evaluations:")
automatic_eval_df.head(1)


Rows after automatic evaluations:


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,word_counts,cost_value,fluency,grammar,tone,length,grounding,latency,cost,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,Take your mobile experience to the next level ...,4720,200,110,87,0.000011,,,,good,,good,good,


In [51]:
# # One entry per product_name in automatic_eval_df (edit defaults / overrides after review).
# _MANUAL_RATING_DEFAULTS: dict[str, str] = {
#     "fluency": "ok",
#     "grammar": "ok",
#     "tone": "ok",
#     "grounding": "ok",
# }
# MANUAL_RATINGS: dict[str, dict[str, str]] = {
#     name: {**_MANUAL_RATING_DEFAULTS}
#     for name in automatic_eval_df["product_name"].unique()
# }

In [70]:
MANUAL_RATINGS_TEMP = {
    "Apple iPhone 15 Pro": {
        "fluency": "ok",
        "grammar": "good",
        "tone": "ok",
        "grounding": "ok",
    },
    "Samsung Galaxy S24 Ultra": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",
    },
    "Google Pixel 8 Pro": {
        "fluency": "ok",
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",
    },
    "Sony WH-1000XM5 Headphones": {
        "fluency": "ok",
        "grammar": "good",
        "tone": "bad",
        "grounding": "ok",
    },
    "Bose QuietComfort Ultra Earbuds": {
        "fluency": "good",
        "grammar": "good",
        "tone": "ok",
        "grounding": "bad",
    },
    "Amazon Echo Dot (5th Gen)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "ok",
        "grounding": "ok",
    },
    "Dell XPS 13 9310 Laptop": {
        "fluency": "ok",
        "grammar": "good",
        "tone": "ok",
        "grounding": "ok",
    },
    "Apple MacBook Air 13″ (M3)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "ok",
        "grounding": "ok",
    },
    "Microsoft Surface Pro 10": {
        "fluency": "ok",
        "grammar": "ok",
        "tone": "ok",
        "grounding": "ok",
    },
    "Garmin Forerunner 255": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",
    },
}

In [ ]:
def merge_manual_ratings(df: pd.DataFrame, manual_ratings: dict) -> pd.DataFrame:
    # Merge the manual_ratings dict into automatic_eval_df
    for product_name, ratings in manual_ratings.items():
        df.loc[df["product_name"] == product_name, "fluency"] = ratings["fluency"]
        df.loc[df["product_name"] == product_name, "grammar"] = ratings["grammar"]
        df.loc[df["product_name"] == product_name, "tone"] = ratings["tone"]
        df.loc[df["product_name"] == product_name, "grounding"] = ratings["grounding"]

    return df


automatic_eval_df = merge_manual_ratings(automatic_eval_df, MANUAL_RATINGS_TEMP)

### 3.3 Final Score

**Calculate `final_score` (pass/fail) using the formula from Task 1**

In [72]:
def calculate_final_score(df: pd.DataFrame) -> pd.DataFrame:
    criteria = [c for c in EVALUATION_CRITERIA if c in df.columns]

    if not criteria:
        print("No criteria to evaluate")
        return df

    def _row_has_all_criteria(row: pd.Series) -> bool:
        for col in criteria:
            val = row[col]

            if pd.isna(val):
                return False

            if isinstance(val, str) and val.strip() == "":
                return False

        return True

    complete_mask = df.apply(_row_has_all_criteria, axis=1)

    for idx in df.index[complete_mask]:
        ratings = {c: df.at[idx, c] for c in criteria}
        df.at[idx, "final_score"] = calculate_pass_fail(ratings)

    return df


final_score_df = calculate_final_score(automatic_eval_df)

In [73]:
# save final_score_df to excel
final_score_df.to_excel(OUTPUT_EXCEL_PATH, index=False, na_rep="")
final_score_df.head(1)

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,word_counts,cost_value,fluency,grammar,tone,length,grounding,latency,cost,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,Take your mobile experience to the next level ...,4720,200,110,87,0.000011,ok,good,ok,good,ok,good,good,pass


### 3.4 Baseline Analysis

**Document your findings**

Based on the manual evaluation:

1. **Best performing criteria:**
   - `length` - Maybe becasue it's the simplest to enforce (rigid rule), AND I added the `max_completion_tokens=150` param with the call.
   - `cost` - Maybe I set too-loose restrictions, but also, this is a simpler model and task so it's reasonable every run is cheap.
   - In terms of quality checks, `grammar` is almost perfect. This is probably since LLMs are trained on massive amount of text, specifically in English, so they provide no typos or nonsense.

2. **Worst performing criteria:**
   - `grounding` - The outputs include exaggerated adjectives (e.g., "escape to serenity" "unlock the ultimate productivity experience!")

3. **Strategy for improvement (Task 4):**
   - To improve the `grounding` issues, we can expand the system prompt to include something like "Use a simple tone - don't add any superlatives. Also, stick ONLY to the provided info and don't add anything in your generated description.", and we can also play with the temperature if applicable (lower == less creative).

---
## Task 4: Improvement Cycle (15 points)

Iterate to achieve better results based on Task 3 baseline analysis.

### Experiment Template

For each experiment, document:
1. **What you changed**
2. **Why you expected it to help**
3. **New evaluation scores**

Keep code for successful experiments. Document failed experiments but code is optional.

In [74]:
def apply_task23_metrics(
    df: pd.DataFrame,
    *,
    n_rows_for_auto_eval: int = None,
    clear_manual_criteria: bool = False,
    price_per_1k_input: float = None,
    price_per_1k_output: float = None,
) -> pd.DataFrame:
    """
    Apply Task 2/3 metrics to a DataFrame with generated descriptions.

    This function:
    1. Computes cost_value from token counts
    2. Evaluates automatic criteria (length, latency, cost)
    3. Optionally clears manual criteria columns for re-evaluation

    Args:
        df: DataFrame with generated descriptions and token counts
        n_rows_for_auto_eval: Number of rows to evaluate (None = all rows)
        clear_manual_criteria: If True, clear manual criteria columns
                               (fluency, grammar, tone, grounding, final_score)
        price_per_1k_input: Price per 1K input tokens (defaults to PRICE_PER_1K_INPUT_TOKENS)
        price_per_1k_output: Price per 1K output tokens (defaults to PRICE_PER_1K_OUTPUT_TOKENS)

    Returns:
        DataFrame with cost_value and automatic evaluations added
    """
    df = df.copy()

    # Use provided pricing or fall back to global variables
    input_price = (
        price_per_1k_input
        if price_per_1k_input is not None
        else PRICE_PER_1K_INPUT_TOKENS
    )
    output_price = (
        price_per_1k_output
        if price_per_1k_output is not None
        else PRICE_PER_1K_OUTPUT_TOKENS
    )

    # Compute cost
    df["cost_value"] = (
        df["input_tokens"] / 1000 * input_price
        + df["output_tokens"] / 1000 * output_price
    )

    # Apply automatic criteria evaluation
    df = evaluate_automatic_criteria(df, n_rows=n_rows_for_auto_eval)

    # Optionally clear manual criteria for re-rating
    if clear_manual_criteria:
        manual_cols = ["fluency", "grammar", "tone", "grounding", "final_score"]
        rows_idx = (
            df.head(n_rows_for_auto_eval).index if n_rows_for_auto_eval else df.index
        )
        for col in manual_cols:
            if col in df.columns:
                df.loc[rows_idx, col] = ""

    return df

### Experiment 1: Prompt engineering

**What changed:**
- Rewrite the system prompt to enforce stricter constraints

**Why expected to help:**
- `grounding` received not-good-enough results due to exaggerated adjectives in the generated description. By adding "Use a simple tone - don't add any superlatives. Also, stick ONLY to the provided info and don't add anything in your generated description" - we expect more relaxed outputs.

**Results:**
- MUCH BETTER descriptions. Overall improvement for almost all relevant outputs. This new, more restrictive and specific system prompt helped.

In [ ]:
# Experiment 1: Stricter prompt for better grounding

EXP1_SYSTEM_PROMPT = """
You are an expert e-commerce copywriter. Your task is to write product descriptions for online shoppers.

CRITICAL REQUIREMENTS:
- Length: Exactly 50-90 words
- Tone: Simple, clear, and factual - NO superlatives or hype words
- Content: Use ONLY the provided product information - do NOT fabricate, infer, or exaggerate any features
- Style: Natural, easy-to-read sentences
- Grammar: Perfect spelling and punctuation

Avoid words like: ultimate, best, revolutionary, amazing, incredible, perfect, etc.
Stick to the facts provided. Make it appealing through clarity, not exaggeration.

OUTPUT: Provide only the product description text. Do not include any preamble or explanation.
""".strip()

# Configuration
EXP1_N_ROWS = 10  # Same as baseline for fair comparison
# EXP1_TEMPERATURE = None  # Lower temperature for more deterministic output
EXP1_MAX_COMPLETION_TOKENS = 150
EXP1_OUTPUT_PATH = "assignment_01_task4_exp1.xlsx"

# Generate descriptions with new prompt
exp1_df = generate_descriptions_for_df(
    df,
    n_rows=EXP1_N_ROWS,
    max_completion_tokens=EXP1_MAX_COMPLETION_TOKENS,
    system_prompt=EXP1_SYSTEM_PROMPT,
)

# Add blank columns for evaluation criteria
for criterion in EVALUATION_CRITERIA:
    exp1_df[criterion] = ""

exp1_df["final_score"] = ""

# Apply metrics
exp1_df = apply_task23_metrics(
    exp1_df,
    n_rows_for_auto_eval=EXP1_N_ROWS,
    clear_manual_criteria=True,  # Clear for manual re-evaluation
)
exp1_df.head(2)

In [ ]:
# exp1_df = pd.read_excel(EXP1_OUTPUT_PATH)
# exp1_df.head(1)

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,cost_value,word_counts,length,latency,cost,fluency,grammar,tone,grounding,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,The Apple iPhone 15 Pro is designed for perfor...,2862,223,104,0.000011,82,good,good,good,good,good,good,good,pass


In [77]:
MANUAL_RATINGS_EXP1 = {
    "Apple iPhone 15 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Samsung Galaxy S24 Ultra": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "bad",  #  `Its 6.8" 120 Hz AMOLED display` - no 6.8" mentioned
    },
    "Google Pixel 8 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Sony WH-1000XM5 Headphones": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Bose QuietComfort Ultra Earbuds": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",
    },
    "Amazon Echo Dot (5th Gen)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Dell XPS 13 9310 Laptop": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Apple MacBook Air 13″ (M3)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Microsoft Surface Pro 10": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Garmin Forerunner 255": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
}

exp1_df = merge_manual_ratings(exp1_df, MANUAL_RATINGS_EXP1)
exp1_df = calculate_final_score(exp1_df)
exp1_df.to_excel(EXP1_OUTPUT_PATH, index=False, na_rep="")

### Experiment 2: Decoding parameters

**What changed:**
- Lowered temperature (lower == less creative).

**Why expected to help:**
- The LLM won't add inforamtion not from the original dataset.

**Results:**
- Best yet (among original run + 1st experiment above) - all reviews got a passing score :)

In [ ]:
# Experiment 2: Lower temperature only (keep original prompt)

# Configuration
EXP2_N_ROWS = 10
EXP2_TEMPERATURE = 0.2  # Very low temperature for minimal creativity
EXP2_OUTPUT_PATH = "assignment_01_exp2.xlsx"

# Generate descriptions with lower temperature
exp2_df = generate_descriptions_for_df(
    df,
    n_rows=EXP2_N_ROWS,
    system_prompt=SYSTEM_PROMPT,  # Use original prompt
    temperature=EXP2_TEMPERATURE,
)

# Apply metrics
exp2_df = apply_task23_metrics(
    exp2_df,
    n_rows_for_auto_eval=EXP2_N_ROWS,
    clear_manual_criteria=True,
)

# Save results
exp2_df.fillna("").to_excel(EXP2_OUTPUT_PATH, index=False, na_rep="")
print(f"\nExperiment 2 results saved to: {EXP2_OUTPUT_PATH}")
print("\nSample results:")
print(exp2_df.head(2))

Processing 1/10: Apple iPhone 15 Pro
Processing 2/10: Samsung Galaxy S24 Ultra
Processing 3/10: Google Pixel 8 Pro
Processing 4/10: Sony WH-1000XM5 Headphones
Processing 5/10: Bose QuietComfort Ultra Earbuds
Processing 6/10: Amazon Echo Dot (5th Gen)
Processing 7/10: Dell XPS 13 9310 Laptop
Processing 8/10: Apple MacBook Air 13″ (M3)
Processing 9/10: Microsoft Surface Pro 10
Processing 10/10: Garmin Forerunner 255

Generation complete!

Experiment 2 results saved to: assignment_01_exp2.xlsx

Sample results:
               product_name  word_counts length latency  cost
0       Apple iPhone 15 Pro         75.0   good    good  good
1  Samsung Galaxy S24 Ultra         84.0   good    good  good
2        Google Pixel 8 Pro         78.0   good    good  good


In [81]:
MANUAL_RATINGS_EXP2 = {
    "Apple iPhone 15 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Samsung Galaxy S24 Ultra": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",
    },
    "Google Pixel 8 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Sony WH-1000XM5 Headphones": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Bose QuietComfort Ultra Earbuds": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",  # `Advanced Noise Cancellation (ANC)` - it's supposed to be `Active Noise Cancellation (ANC)`
    },
    "Amazon Echo Dot (5th Gen)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Dell XPS 13 9310 Laptop": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Apple MacBook Air 13″ (M3)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Microsoft Surface Pro 10": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Garmin Forerunner 255": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
}

exp2_df = merge_manual_ratings(exp2_df, MANUAL_RATINGS_EXP2)
exp2_df = calculate_final_score(exp2_df)
exp2_df.to_excel(EXP2_OUTPUT_PATH, index=False, na_rep="")

### Experiment 3: Combined Approach

**What changed:**
- Combined stricter prompt (from Exp 1) with lower temperature (0.2, from Exp 2)

**Why expected to help:**
- The combination of explicit grounding instructions and reduced sampling randomness should maximize factual accuracy while maintaining natural language quality.

**Results:**
- PERFECT. All processed items recieved the highest, `good`, result for each criterion.

In [ ]:
# Experiment 3: Combined approach (stricter prompt + lower temperature)

# Configuration
EXP3_N_ROWS = 10
EXP3_OUTPUT_PATH = "assignment_01_exp3.xlsx"

# Generate descriptions with both improvements
exp3_df = generate_descriptions_for_df(
    df,
    n_rows=EXP3_N_ROWS,
    system_prompt=EXP1_SYSTEM_PROMPT,  # Use stricter prompt from Exp 1
    temperature=EXP2_TEMPERATURE,  # same as Exp 2
)

# Apply metrics
exp3_df = apply_task23_metrics(
    exp3_df,
    n_rows_for_auto_eval=EXP3_N_ROWS,
    clear_manual_criteria=True,
)

# Save results
exp3_df.fillna("").to_excel(EXP3_OUTPUT_PATH, index=False, na_rep="")
print(f"\nExperiment 3 results saved to: {EXP3_OUTPUT_PATH}")
print("\nSample results:")
print(exp3_df.head(2))

Processing 1/10: Apple iPhone 15 Pro
Processing 2/10: Samsung Galaxy S24 Ultra
Processing 3/10: Google Pixel 8 Pro
Processing 4/10: Sony WH-1000XM5 Headphones
Processing 5/10: Bose QuietComfort Ultra Earbuds
Processing 6/10: Amazon Echo Dot (5th Gen)
Processing 7/10: Dell XPS 13 9310 Laptop
Processing 8/10: Apple MacBook Air 13″ (M3)
Processing 9/10: Microsoft Surface Pro 10
Processing 10/10: Garmin Forerunner 255

Generation complete!

Experiment 3 results saved to: assignment_01_exp3.xlsx

Sample results:
          product_name  word_counts length latency  cost
0  Apple iPhone 15 Pro         68.0   good      ok  good


In [84]:
MANUAL_RATINGS_EXP3 = {
    "Apple iPhone 15 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Samsung Galaxy S24 Ultra": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Google Pixel 8 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Sony WH-1000XM5 Headphones": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Bose QuietComfort Ultra Earbuds": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Amazon Echo Dot (5th Gen)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Dell XPS 13 9310 Laptop": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Apple MacBook Air 13″ (M3)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Microsoft Surface Pro 10": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Garmin Forerunner 255": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
}

exp3_df = merge_manual_ratings(exp3_df, MANUAL_RATINGS_EXP3)
exp3_df = calculate_final_score(exp3_df)
exp3_df.to_excel(EXP3_OUTPUT_PATH, index=False, na_rep="")

---
## Task 5: Create a Judge Model (20 points)

Build an automated LLM judge that grades descriptions using the Task 1 rubric.

### 5.1 Judge Model Selection

Start with the model you **did not** use in Task 2. If it struggles, switch to a larger model.

In [17]:
# Choose judge model (the one NOT used in Task 2)
JUDGE_MODEL_NAME = "google/gemma-2-9b-it"  # or another model if needed

print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Generator model was: {MODEL_NAME}")

Judge model: google/gemma-2-9b-it
Generator model was: meta-llama/Meta-Llama-3.1-8B-Instruct


### 5.2 Pydantic Schema for Structured Output

Define the output schema. Note: **explanation comes before verdict** (important for chain-of-thought reasoning).

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field  # , create_model


class CriterionEvaluation(BaseModel):
    explanation: str = Field(description="Reasoning for the verdict")
    verdict: Literal["good", "ok", "bad"] = Field(
        description="Rating: good, ok, or bad"
    )


class DescriptionEvaluation(BaseModel):
    fluency: CriterionEvaluation
    grammar: CriterionEvaluation
    tone: CriterionEvaluation
    length: CriterionEvaluation
    grounding: CriterionEvaluation


# Optional to write it programmatically (less readable)
# DescriptionEvaluation = create_model(
#     "DescriptionEvaluation",
#     **{name: (CriterionEvaluation, ...) for name in QUALITY_CRITERIA},
# )

# Display schema
print("Judge output schema:")
print(DescriptionEvaluation.model_json_schema())

Judge output schema:
{'$defs': {'CriterionEvaluation': {'properties': {'explanation': {'description': 'Reasoning for the verdict', 'title': 'Explanation', 'type': 'string'}, 'verdict': {'description': 'Rating: good, ok, or bad', 'enum': ['good', 'ok', 'bad'], 'title': 'Verdict', 'type': 'string'}}, 'required': ['explanation', 'verdict'], 'title': 'CriterionEvaluation', 'type': 'object'}}, 'properties': {'fluency': {'$ref': '#/$defs/CriterionEvaluation'}, 'grammar': {'$ref': '#/$defs/CriterionEvaluation'}, 'tone': {'$ref': '#/$defs/CriterionEvaluation'}, 'length': {'$ref': '#/$defs/CriterionEvaluation'}, 'grounding': {'$ref': '#/$defs/CriterionEvaluation'}}, 'required': ['fluency', 'grammar', 'tone', 'length', 'grounding'], 'title': 'DescriptionEvaluation', 'type': 'object'}


**Why explanation before verdict?**

`explanation` comes before `verdict` in the schema to encourage the LLM to provide rubric-grounded judgments and useful explanations for analysis.

### Why it helps for an LLM judge

- **Think-then-label:**  
  You want the model to ground the rating in the text first, then map that reasoning to good / ok / bad. Declaring explanation first (and often listing it first in the schema) nudges generation toward "reason → verdict" instead of "verdict → justify".

- **Less shallow rationalization:**  
  If verdict were first, some models tend to pick a label early and then write a short, generic explanation that matches it. Putting reasoning first makes that pattern a bit harder and often yields more specific, rubric-linked explanations.

> *Note: this is not a guarantee to provide better results—the model can still tweak its evaluation.*

### 5.3 Judge Prompt

Write a prompt that embeds the Task 1 rubric and provides necessary context for evaluation.

In [ ]:
LLM_JUDGE_EVALUATION_CRITERIA_STRING = ""

for name in QUALITY_CRITERIA:
    LLM_JUDGE_EVALUATION_CRITERIA_STRING += f"{name.lower()}:\n"

    for threshold in ["good", "ok", "bad"]:
        LLM_JUDGE_EVALUATION_CRITERIA_STRING += f'\t- "{threshold}": {CRITERION_THRESHOLDS[name.lower()][threshold]["description"]}\n'

    LLM_JUDGE_EVALUATION_CRITERIA_STRING += "\n"

JUDGE_SYSTEM_PROMPT = f"""
You are an expert evaluator of product descriptions. Your task is to rate product descriptions according to specific criteria.

For each criterion, provide:
1. An explanation of your reasoning
2. A verdict: 'good', 'ok', or 'bad'

EVALUATION CRITERIA:
{LLM_JUDGE_EVALUATION_CRITERIA_STRING}
Be objective and consistent in your evaluations.
""".strip()


def create_judge_prompt(
    description: str, product_name: str, attributes: str, material: str, warranty: str
) -> str:
    return f"""PRODUCT INFORMATION:
Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

GENERATED DESCRIPTION:
{description}

Evaluate this description according to the criteria (fluency, grammar, tone, length, grounding)."""

Judge prompt created


In [57]:
print(JUDGE_SYSTEM_PROMPT)

You are an expert evaluator of product descriptions. Your task is to rate product descriptions according to specific criteria.

For each criterion, provide:
1. An explanation of your reasoning
2. A verdict: 'good', 'ok', or 'bad'

EVALUATION CRITERIA:
fluency:
	- "good": Natural, smooth sentences with varied structure. Easy to read aloud. No awkward phrasing or repetition.
	- "ok": Mostly natural but with minor awkwardness (e.g., one slightly repetitive phrase or choppy transition).
	- "bad": Multiple awkward phrases, unnatural word order, or repetitive structure that disrupts readability.

grammar:
	- "good": Zero spelling or punctuation errors. Proper sentence structure throughout.
	- "ok": One minor error (e.g., missing comma, minor typo) that doesn't affect comprehension.
	- "bad": Multiple errors or one major error (e.g., subject-verb disagreement, misspelled product name).

tone:
	- "good": Consistently friendly, credible sales voice. Enthusiastic without being pushy. Professiona

### 5.4 Judge Implementation

In [ ]:
def judge_description(
    description: str, product_name: str, attributes: str, material: str, warranty: str
) -> DescriptionEvaluation:
    """
    Use the judge model to evaluate a product description.

    Returns:
        DescriptionEvaluation object with ratings for each criterion
    """
    user_prompt = create_judge_prompt(
        description, product_name, attributes, material, warranty
    )

    response = client.chat.completions.create(
        model=JUDGE_MODEL_NAME,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,  # Lower temperature for more consistent evaluation
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "description_evaluation",
                "schema": DescriptionEvaluation.model_json_schema(),
            },
        },
    )

    content = response.choices[0].message.content

    # Parse into Pydantic model
    return DescriptionEvaluation.model_validate_json(content)

Judge function ready


---
## Task 6: Run and Analyze the Judge (20 points)

Run the judge model and compare its evaluations to human ratings.

### 6.1 Sanity Check (5 products)

In [ ]:
# TODO: Run judge on 5 products for sanity check
results_df = pd.read_excel(OUTPUT_EXCEL_PATH)

# Select 5 products (can be random or specific)
sanity_check_indices = [0, 10, 20, 30, 40]  # Adjust as needed

print("Sanity Check - Judge Evaluations:")
print("=" * 80)

for idx in sanity_check_indices:
    row = results_df.iloc[idx]

    print(f"\nProduct: {row['product_name']}")
    print(f"Description: {row['generated_description'][:100]}...")

    evaluation = judge_description(
        description=row["generated_description"],
        product_name=row["product_name"],
        attributes=row["Product_attribute_list"],
        material=row["material"],
        warranty=row["warranty"],
    )

    print("\nJudge Ratings:")
    for criterion in ["fluency", "grammar", "tone", "length", "grounding"]:
        eval_obj = getattr(evaluation, criterion)
        print(f"  {criterion}: {eval_obj.verdict}")
        print(f"    → {eval_obj.explanation}")

    time.sleep(1)  # Rate limiting

print("\n" + "=" * 80)
print("Review the explanations and verdicts above.")
print("Does the judge apply your rubric correctly? Adjust prompt if needed.")

**Sanity Check Analysis:**

[TODO: Review the judge outputs above. Do they make sense? Does the judge apply your rubric correctly? Document any issues and prompt adjustments needed.]

### 6.2 Full Run - Judge All Products

In [ ]:
# TODO: Run judge on all products
results_df = pd.read_excel(OUTPUT_EXCEL_PATH)

# Add judge columns
judge_columns = [
    "judge_fluency",
    "judge_fluency_explanation",
    "judge_grammar",
    "judge_grammar_explanation",
    "judge_tone",
    "judge_tone_explanation",
    "judge_length",
    "judge_length_explanation",
    "judge_grounding",
    "judge_grounding_explanation",
    "judge_final_score",
]

for col in judge_columns:
    if col not in results_df.columns:
        results_df[col] = ""

print(f"Running judge on {len(results_df)} products...")

for idx, row in results_df.iterrows():
    print(f"Judging {idx + 1}/{len(results_df)}: {row['product_name']}")

    evaluation = judge_description(
        description=row["generated_description"],
        product_name=row["product_name"],
        attributes=row["Product_attribute_list"],
        material=row["material"],
        warranty=row["warranty"],
    )

    # Store judge ratings
    for criterion in QUALITY_CRITERIA:
        eval_obj = getattr(evaluation, criterion)
        results_df.at[idx, f"judge_{criterion}"] = eval_obj.verdict
        results_df.at[idx, f"judge_{criterion}_explanation"] = eval_obj.explanation

    # Calculate judge final score using Task 1 formula
    judge_ratings = {
        "fluency": results_df.at[idx, "judge_fluency"],
        "grammar": results_df.at[idx, "judge_grammar"],
        "tone": results_df.at[idx, "judge_tone"],
        "length": results_df.at[idx, "judge_length"],
        "grounding": results_df.at[idx, "judge_grounding"],
        "latency": results_df.at[idx, "latency"],  # From Task 2
        "cost": results_df.at[idx, "cost"],  # From Task 3
    }

    # Apply pass/fail formula from Task 1
    # TODO: Implement calculate_pass_fail function or inline logic
    results_df.at[idx, "judge_final_score"] = "pass"  # Placeholder

    time.sleep(1)  # Rate limiting

# Save updated results
results_df.to_excel(OUTPUT_EXCEL_PATH, index=False)
print(f"\nJudge evaluation complete! Results saved to {OUTPUT_EXCEL_PATH}")

### 6.3 Compare Judge vs Human Evaluation

In [ ]:
# Load results with both human and judge evaluations
results_df = pd.read_excel(OUTPUT_EXCEL_PATH)

# Filter rows with human evaluation
compared = results_df[results_df["fluency"] != ""].copy()

print(f"Comparing judge vs human on {len(compared)} products\n")
print("Agreement Rates by Criterion:")
print("=" * 50)


for criterion in QUALITY_CRITERIA:
    human_col = criterion
    judge_col = f"judge_{criterion}"

    if human_col in compared.columns and judge_col in compared.columns:
        agreements = (compared[human_col] == compared[judge_col]).sum()
        total = len(compared)
        agreement_rate = (agreements / total * 100) if total > 0 else 0

        print(f"\n{criterion.upper()}:")
        print(f"  Agreement: {agreements}/{total} ({agreement_rate:.1f}%)")

        # Show disagreements
        disagreements = compared[compared[human_col] != compared[judge_col]]
        if len(disagreements) > 0:
            print("  Disagreements:")
            for _, row in disagreements.iterrows():
                print(
                    f"    - {row['product_name'][:40]}: Human={row[human_col]}, Judge={row[judge_col]}"
                )

# Overall agreement
print(f"\n{'=' * 50}")
print("Overall Analysis:")
# TODO: Calculate overall agreement and analyze patterns

**Analysis of Judge vs Human Agreement:**

[TODO: Document your findings]

1. **Where do they agree most?**
   - 

2. **Where do they diverge?**
   - 

3. **Why might these differences occur?**
   - 

### 6.4 Criterion-by-Criterion Judging

Run the judge separately for each criterion (one API call per criterion per product).

In [ ]:
# TODO: Implement single-criterion judge
def judge_single_criterion(
    description: str,
    product_name: str,
    attributes: str,
    material: str,
    warranty: str,
    criterion: str,
) -> CriterionEvaluation:
    """
    Judge a single criterion in isolation.

    Args:
        criterion: One of 'fluency', 'grammar', 'tone', 'length', 'grounding'
    """
    # Create criterion-specific prompt
    criterion_prompt = f"""PRODUCT INFORMATION:
Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

GENERATED DESCRIPTION:
{description}

Evaluate ONLY the {criterion.upper()} of this description according to the rubric."""

    # TODO: Implement API call for single criterion
    # Similar to judge_description but returns only CriterionEvaluation
    pass


# Run criterion-by-criterion evaluation on a subset
# TODO: Implement and compare results

**Criterion-by-Criterion Analysis:**

[TODO: Answer these questions]

1. **Did isolating criteria change the results?**
   - 

2. **Why might this approach lead to different outcomes?**
   - 

3. **Did agreement with human scores improve?**
   - 

### 6.5 Final Analysis and Reflection

#### Question 1: Trade-offs between human evaluation and LLM-as-a-judge

Consider: cost, scale, consistency, accuracy

[TODO: Write your analysis here]

**Human Evaluation:**
- Pros:
  - 
- Cons:
  - 

**LLM-as-a-Judge:**
- Pros:
  - 
- Cons:
  - 

#### Question 2: Recommendation for production system

For a production system generating thousands of descriptions daily:

[TODO: Write your recommendation here]

**Recommended approach:**
- 

**Justification:**
- 

**Implementation considerations:**
- 

---
## Summary and Submission

### Deliverables Checklist

- [ ] Task 1: Rubric definitions and pass/fail formula
- [ ] Task 2: Code for description generation
- [ ] Task 3: `assignment_01.xlsx` with manual evaluations (10-15 products)
- [ ] Task 4: Experiment documentation + code for successful experiments
- [ ] Task 5: Judge model implementation with Pydantic schema
- [ ] Task 6: Judge analysis, comparisons, and reflection

### Files to Submit

1. `assignment_01_solution.ipynb` (this notebook)
2. `assignment_01.xlsx` (with all evaluations)

**Due Date:** April 5, 2026